<a href="https://colab.research.google.com/github/gyeongee/TIL/blob/main/%ED%8C%8C%EC%9D%B4%EC%8D%AC_%EC%B6%94%EC%B2%9C_%EC%8B%9C%EC%8A%A4%ED%85%9C%ED%8C%A8%ED%82%A4%EC%A7%80_Surprise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

학습 데이터로  fit()을 호출해 학습한 뒤 테스트 데이터로 test()를 호출해 예측 평점을 계산하고 MSE/RMSE로 성능 평가해보겠습니다.

In [ ]:
!pip install scikit-surprise

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 9.3 MB/s  0:00:00


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from surprise import SVD,Dataset,accuracy,Reader
from surprise.model_selection import train_test_split

Surprise는 무비렌즈 사이트에서 제공하는 과거 버전의 데이터 세트를 가져오는 API를 제공합니다.

In [ ]:
data = Dataset.load_builtin('ml-100k')
# 수행 시마다 동일하게 데이터를 분할라기 위해 random_state 값 부여
trainset,testset=train_test_split(data,test_size=.25,random_state=0)

Dataset ml-100k could not be found. Do you want to download it? [Y/n] y
Trying to download dataset from https://files.grouplens.org/datasets/movielens/ml-100k.zip...


URLError: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1032)>

Colab의 Python 3.13 환경에서 SSL 인증서를 제대로 찾지 못해서 MovieLens 데이터를 다운로드하지 못하는 문제입니다.

직접 다운로드해서 압축을 풀어줍니다.

네. MovieLens 100K(ml-100k)의 원본 데이터는 기본적인 데이터 정제가 되어있습니다.

In [ ]:
!wget --no-check-certificate https://files.grouplens.org/datasets/movielens/ml-100k.zip

--2026-10-26 12:46:37--  https://files.grouplens.org/datasets/movielens/ml-100k.zip
Resolving files.grouplens.org (files.grouplens.org)... 128.101.96.204
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:443... connected.
  Unable to locally verify the issuer's authority.
HTTP request sent, awaiting response... 200 OK
Length: 4924029 (4.7M) [application/zip]
Saving to: ‘ml-100k.zip’

ml-100k.zip         100%[===================>]   4.70M  3.53MB/s    in 1.3s    

2026-10-26 12:46:39 (3.53 MB/s) - ‘ml-100k.zip’ saved [4924029/4924029]



In [ ]:
!unzip '/content/ml-100k.zip'

Archive:  /content/ml-100k.zip
   creating: ml-100k/
  inflating: ml-100k/allbut.pl       
  inflating: ml-100k/mku.sh          
  inflating: ml-100k/README          
  inflating: ml-100k/u.data          
  inflating: ml-100k/u.genre         
  inflating: ml-100k/u.info          
  inflating: ml-100k/u.item          
  inflating: ml-100k/u.occupation    
  inflating: ml-100k/u.user          
  inflating: ml-100k/u1.base         
  inflating: ml-100k/u1.test         
  inflating: ml-100k/u2.base         
  inflating: ml-100k/u2.test         
  inflating: ml-100k/u3.base         
  inflating: ml-100k/u3.test         
  inflating: ml-100k/u4.base         
  inflating: ml-100k/u4.test         
  inflating: ml-100k/u5.base         
  inflating: ml-100k/u5.test         
  inflating: ml-100k/ua.base         
  inflating: ml-100k/ua.test         
  inflating: ml-100k/ub.base         
  inflating: ml-100k/ub.test         


In [ ]:
import pandas as pd

ratings = pd.read_csv(
    '/content/ml-100k/u.data',
    sep='\t',
    header=None,
    names=['userId', 'movieId', 'rating', 'timestamp']
)

ratings.head()

,userId,movieId,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


In [ ]:
# 데이터 평점의 범위
reader = Reader(rating_scale=(1, 5))

data = Dataset.load_from_df(
    ratings[['userId', 'movieId', 'rating']],
    reader
)

In [ ]:
data

In [ ]:
trainset,testset=train_test_split(data,test_size=.25,random_state=0)

In [ ]:
algo=SVD(random_state=0)
algo.fit(trainset)

- test(): 사용자-아이템 평점 데이터 세트 전체에 대해서 추천을 예측하는 메서드

- predict(): 개별 사용자와 영화에 대한 추천 평점 반환

In [ ]:
predictions=algo.test(testset)
print(f'prediction type:{type(predictions)},size:{len(predictions)}')
print(f'prediction 결과의 최초 5개 추출')
predictions[:5]


prediction type:<class 'list'>,size:25000
prediction 결과의 최초 5개 추출


[Prediction(uid=120, iid=282, r_ui=4.0, est=np.float64(3.5114147666251547), details={'was_impossible': False}),
 Prediction(uid=882, iid=291, r_ui=4.0, est=np.float64(3.573872419581491), details={'was_impossible': False}),
 Prediction(uid=535, iid=507, r_ui=5.0, est=np.float64(4.033583485472447), details={'was_impossible': False}),
 Prediction(uid=697, iid=244, r_ui=5.0, est=np.float64(3.8463639495936905), details={'was_impossible': False}),
 Prediction(uid=751, iid=385, r_ui=4.0, est=np.float64(3.1807542478219157), details={'was_impossible': False})]

SVD 알고리즘 객체의 test(데이터 세트) 메서드의 호출 결과는 파이썬 리스트이며 크기는 입력 인자 데이터 세트의 크기와 같은 25,000개입니다.

호출 결과로 반환된 리스트 객체는 25,00개의 Predition 객체를 내부에 가지고 있습니다. Prediction 객체는 Surprise 패키지에서 제공하는 데이터 타입이며, 개별 사용자 아이디(uid), 영화 (또는 아이템) 아이디(iid)와 실제 평점(r_ui) 정보에 기반해 Surprise의 추천 예측 평점(est)데이터를 튜플 형태로 가지고 있습ㄴ디ㅏ.

Prediction 객체의 details 속성은 내부 처리 시 추천 예측을 할 수 없는 경우에 로그용으로 데이터를 남기는 데 사용됩니다.

`was_impossible`이 `True`이면 예측값을 생성할 수 없는 데이터라는 의미입니다. 여기서는 모두 False입니다.



`predict()`는 개별 사용자의 아이템에 대한 추천 평점을 예측해 줍니다.

In [ ]:
# 사용자 아이디, 아이템 아이디는 문자열로 입력해야 함
uid=str(196)
iid=str(302)
pred=algo.predict(uid,iid)
print(pred)

user: 196        item: 302        r_ui = None   est = 3.53   {'was_impossible': False}


결과처럼 predict()는 개별 사용자와 아이템 정보를 입력하면 추천 예측 평점 est로 반환합니다.

test() 메서드는 입력 데이터 세트의 모든 사용자와 아이템 아이디에 대해서 predict()를 반복적으로 수행한 결과라고 생각하면 좀 더 이해하기 쉬울 것입니다.

In [ ]:
# 추천 시스템의 성능 평가 정보
accuracy.rmse(predictions)

RMSE: 0.9467


np.float64(0.9466860806937948)